<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/03_deep_learning/architectures/cnn/experiment_cnn_scratch_vs_transfer_learning_resnet18_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# LEVEL 1: TRAIN FROM SCRATCH
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Data
transform = transforms.ToTensor()

train_data = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=64)

# Simple CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 16, 3)
        self.fc = nn.Linear(16*26*26, 10)

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model_scratch = SimpleCNN()

optimizer = optim.Adam(model_scratch.parameters())
loss_fn = nn.CrossEntropyLoss()

for epoch in range(2):
    for images, labels in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model_scratch(images), labels)
        loss.backward()
        optimizer.step()

print("Scratch training done")

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 340kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.17MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.57MB/s]


Scratch training done


In [2]:
# ============================================
# LEVEL 2: PRETRAINED MODEL
# ============================================

from torchvision import models

model_pretrained = models.resnet18(pretrained=True)

print("Loaded pretrained model")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 192MB/s]


Loaded pretrained model


In [3]:
# Freeze layers
for param in model_pretrained.parameters():
    param.requires_grad = False

# Replace final layer
model_pretrained.fc = nn.Linear(model_pretrained.fc.in_features, 10)

In [4]:
optimizer = optim.Adam(model_pretrained.fc.parameters())

for epoch in range(2):
    for images, labels in train_loader:

        optimizer.zero_grad()

        output = model_pretrained(images.repeat(1,3,1,1))  # MNIST → 3 channel

        loss = loss_fn(output, labels)

        loss.backward()
        optimizer.step()

print("Fine-tuning done")

Fine-tuning done


In [5]:
for param in model_pretrained.layer4.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model_pretrained.parameters(), lr=1e-4)

In [6]:
print("Scratch vs Transfer Learning")

print("Scratch → slower, needs more data")
print("Transfer → faster, better accuracy")

Scratch vs Transfer Learning
Scratch → slower, needs more data
Transfer → faster, better accuracy
